In [ ]:
# =====================================================================================
#  OCH-MRNet v2: Ordinal Cost-sensitive Hybrid Maternal Risk Network (OOF stacking + cross-conformal)
#  Full experimental pipeline (Q2 journal) — Maternal Health Risk + Fetal Health (CTG)
#  Run cell-by-cell on Kaggle (GPU T4). Every "# %%" block = one notebook cell.
#  Outputs -> /kaggle/working/outputs/{figures,tables}
# =====================================================================================

In [ ]:
# %% [C2] CONFIG + IMPORTS ===========================================================
import os, time, math, json, random, warnings, itertools, copy
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
from scipy.optimize import minimize, minimize_scalar
from sklearn.base import BaseEstimator, ClassifierMixin, TransformerMixin, clone
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold, StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier, AdaBoostClassifier,
                              GradientBoostingClassifier, StackingClassifier)
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_recall_fscore_support,
                             matthews_corrcoef, cohen_kappa_score, roc_auc_score, log_loss,
                             confusion_matrix, roc_curve, precision_recall_curve, auc, f1_score, recall_score)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
warnings.filterwarnings("ignore")

CFG = dict(
    SEED=42,
    OUT="/kaggle/working/outputs",
    MATERNAL_PATH="/kaggle/input/datasets/csafrit2/maternal-health-risk-data/Maternal Health Risk Data Set.csv",
    FETAL_PATH="/kaggle/input/datasets/andrewmvd/fetal-health-classification/fetal_health.csv",
    N_SPLITS=5,
    N_REPEATS={"maternal": 5, "fetal": 3},   # Protocol B (leakage-free) repeats
    N_REPEATS_A=3,                           # Protocol A (naive, literature-style) repeats
    RUN_FETAL=True,
    RUN_ABLATION=True,
    ABL_REPEATS={"maternal": 2, "fetal": 1},   # repeats on which retrained ablations run
    INNER_FOLDS={"maternal": 4, "fetal": 3},   # inner CV for out-of-fold stacking
    HIGH_COST=2.0,                           # extra cost on the highest-risk class
    CONF_ALPHAS=[0.05, 0.10, 0.15, 0.20],
    MAIN_ALPHA=0.10,
    NOISE_LEVELS=[0.0, 0.05, 0.10, 0.20, 0.30],
    FT=dict(d_token=32, n_layers=3, n_heads=4, dropout=0.15, lr=1e-3, wd=1e-4,
            batch=64, max_epochs=400, patience=40),
    FAST_MODE=False,                         # True = quick dry run (1 repeat, fewer epochs)
)
if CFG["FAST_MODE"]:
    CFG["N_REPEATS"] = {"maternal": 1, "fetal": 1}; CFG["N_REPEATS_A"] = 1; CFG["ABL_REPEATS"] = {"maternal": 1, "fetal": 1}
    CFG["FT"].update(max_epochs=60, patience=10)

FIG = os.path.join(CFG["OUT"], "figures"); TAB = os.path.join(CFG["OUT"], "tables")
os.makedirs(FIG, exist_ok=True); os.makedirs(TAB, exist_ok=True)

sns.set_theme(style="whitegrid", context="paper", font_scale=1.15)
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "axes.titleweight": "bold"})
PROP = "OCH-MRNet (Proposed)"
PROP_COLOR, BASE_COLOR = "#C0392B", "#5D6D7E"

def set_seed(s):
    random.seed(s); np.random.seed(s)
    try:
        import torch; torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    except Exception: pass

def savefig(fig, name):
    fig.tight_layout()
    fig.savefig(os.path.join(FIG, f"{name}.png"), bbox_inches="tight")
    fig.savefig(os.path.join(FIG, f"{name}.pdf"), bbox_inches="tight")
    plt.show(); plt.close(fig)
    print(f"[saved figure] {name}")

def save_table(df, name, index=False):
    df.to_csv(os.path.join(TAB, f"{name}.csv"), index=index)
    try:
        df.to_latex(os.path.join(TAB, f"{name}.tex"), index=index, escape=True)
    except Exception as e:
        print("  (latex export skipped:", e, ")")
    print(f"[saved table] {name}  shape={df.shape}")

set_seed(CFG["SEED"])
print("Config ready. Output dir:", CFG["OUT"])

In [ ]:
# %% [C3] DATA LOADING + DUPLICATE / CONFLICT AUDIT ==================================
def load_maternal(path):
    df = pd.read_csv(path); df.columns = [c.strip() for c in df.columns]
    df["RiskLevel"] = df["RiskLevel"].astype(str).str.strip().str.lower()
    y = df["RiskLevel"].map({"low risk": 0, "mid risk": 1, "high risk": 2})
    assert y.notna().all(), "Unexpected RiskLevel labels"
    return df.drop(columns="RiskLevel").astype(float), y.astype(int)

def load_fetal(path):
    df = pd.read_csv(path); df.columns = [c.strip().replace(" ", "_") for c in df.columns]
    y = df["fetal_health"].astype(int) - 1
    return df.drop(columns="fetal_health").astype(float), y

def build_dataset(name, X, y, class_names):
    full = X.copy(); full["__y"] = y.values
    dup_mask = full.duplicated()
    d = full.loc[~dup_mask].reset_index(drop=True)
    Xd, yd = d.drop(columns="__y"), d["__y"].astype(int).values
    groups = pd.factorize(pd.util.hash_pandas_object(Xd, index=False))[0]   # identical feature vectors -> same group
    n_label_per_vec = pd.Series(yd).groupby(groups).nunique()
    conflict_groups = int((n_label_per_vec > 1).sum())
    rows_in_conflict = int(pd.Series(groups).isin(n_label_per_vec[n_label_per_vec > 1].index).sum())
    best_per_vec = pd.Series(yd).groupby(groups).agg(lambda s_: s_.value_counts().iloc[0])
    ceiling = float(best_per_vec.sum() / len(yd))      # max accuracy any model can reach on dedup data
    audit = dict(Dataset=name, Raw_rows=len(X), Exact_duplicates=int(dup_mask.sum()),
                 Duplicate_pct=round(dup_mask.mean()*100, 2), Unique_rows=len(Xd),
                 Unique_feature_vectors=int(len(np.unique(groups))),
                 Conflicting_label_groups=conflict_groups, Rows_in_conflict_groups=rows_in_conflict,
                 Label_noise_accuracy_ceiling=round(ceiling, 4))
    for k, c in enumerate(class_names):
        audit[f"Raw_{c}"] = int((y.values == k).sum()); audit[f"Dedup_{c}"] = int((yd == k).sum())
    return dict(name=name, X=X.reset_index(drop=True), y=y.values, Xd=Xd, yd=yd, gd=groups,
                classes=class_names, K=len(class_names), high=len(class_names)-1, audit=audit)

DATA = {}
Xm, ym = load_maternal(CFG["MATERNAL_PATH"])
DATA["maternal"] = build_dataset("Maternal Health Risk", Xm, ym, ["Low", "Mid", "High"])
if CFG["RUN_FETAL"]:
    Xf, yf = load_fetal(CFG["FETAL_PATH"])
    DATA["fetal"] = build_dataset("Fetal Health (CTG)", Xf, yf, ["Normal", "Suspect", "Pathological"])

audit_df = pd.DataFrame([D["audit"] for D in DATA.values()]).T.reset_index()
audit_df.columns = ["Item"] + [D["name"] for D in DATA.values()]
save_table(audit_df, "T02_duplicate_audit")
print(audit_df.to_string(index=False))

# Table 1: dataset description
desc_rows = []
for key, D in DATA.items():
    for c in D["Xd"].columns:
        s = D["Xd"][c]
        desc_rows.append(dict(Dataset=D["name"], Feature=c, Mean=round(s.mean(), 3), SD=round(s.std(), 3),
                              Min=s.min(), Median=s.median(), Max=s.max(), Unique=s.nunique()))
save_table(pd.DataFrame(desc_rows), "T01_dataset_description")
print("Implausible HeartRate (<40 bpm) rows in maternal raw data:", int((Xm["HeartRate"] < 40).sum()))

In [ ]:
# %% [C4] CLINICAL KNOWLEDGE FEATURES + FOLD-SAFE PREPROCESSOR ========================
PLAUSIBLE = {"maternal": {"HeartRate": (40, 200), "SystolicBP": (60, 250), "DiastolicBP": (30, 150)},
             "fetal": {}}

def kf_maternal(X):
    X = X.copy()
    X["MAP"] = (X["SystolicBP"] + 2*X["DiastolicBP"]) / 3
    X["PulsePressure"] = X["SystolicBP"] - X["DiastolicBP"]
    X["ShockIndex"] = X["HeartRate"] / X["SystolicBP"]
    X["BP_Ratio"] = X["SystolicBP"] / X["DiastolicBP"]
    X["HTN"] = ((X["SystolicBP"] >= 140) | (X["DiastolicBP"] >= 90)).astype(float)
    X["SevereHTN"] = ((X["SystolicBP"] >= 160) | (X["DiastolicBP"] >= 110)).astype(float)
    X["HyperGly"] = (X["BS"] >= 7.8).astype(float)
    X["DiabRange"] = (X["BS"] >= 11.1).astype(float)
    X["Fever"] = (X["BodyTemp"] >= 100.4).astype(float)
    X["AdvAge"] = (X["Age"] >= 35).astype(float)
    X["Teen"] = (X["Age"] < 20).astype(float)
    X["ClinicalFlagCount"] = X[["HTN", "HyperGly", "Fever", "AdvAge", "Teen"]].sum(1)
    return X

def kf_fetal(X):
    X = X.copy()
    X["TotalDecel"] = X["light_decelerations"] + X["severe_decelerations"] + X["prolongued_decelerations"]
    X["SevereOrProlonged"] = ((X["severe_decelerations"] + X["prolongued_decelerations"]) > 0).astype(float)
    X["AccelPresent"] = (X["accelerations"] > 0).astype(float)
    X["BaselineAbn"] = ((X["baseline_value"] < 110) | (X["baseline_value"] > 160)).astype(float)
    X["HighASTV"] = (X["abnormal_short_term_variability"] > 60).astype(float)
    X["HighALTV"] = (X["percentage_of_time_with_abnormal_long_term_variability"] >= 20).astype(float)
    X["DecelPerContraction"] = X["TotalDecel"] / (X["uterine_contractions"] + 1e-3)
    X["HistSkew"] = X["histogram_mean"] - X["histogram_median"]
    X["ClinicalFlagCount"] = X[["SevereOrProlonged", "BaselineAbn", "HighASTV", "HighALTV"]].sum(1) + (1 - X["AccelPresent"])
    return X

KF_FN = {"maternal": kf_maternal, "fetal": kf_fetal}

class ClinicalPreprocessor(BaseEstimator, TransformerMixin):
    """Fold-safe: plausibility check -> median impute (train stats) -> optional knowledge features -> optional scaling."""
    def __init__(self, dataset="maternal", use_kf=False, scale=True):
        self.dataset, self.use_kf, self.scale = dataset, use_kf, scale
    def _clean(self, X):
        X = X.copy()
        for c, (lo, hi) in PLAUSIBLE[self.dataset].items():
            if c in X: X.loc[(X[c] < lo) | (X[c] > hi), c] = np.nan
        return X
    def fit(self, X, y=None):
        X = pd.DataFrame(X, columns=getattr(self, "cols_", None) or list(X.columns))
        self.cols_ = list(X.columns)
        Xc = self._clean(X); self.medians_ = Xc.median()
        Xc = Xc.fillna(self.medians_)
        if self.use_kf: Xc = KF_FN[self.dataset](Xc)
        self.feature_names_out_ = list(Xc.columns)
        self.scaler_ = StandardScaler().fit(Xc.values) if self.scale else None
        return self
    def transform(self, X):
        X = X if isinstance(X, pd.DataFrame) else pd.DataFrame(X, columns=self.cols_)
        Xc = self._clean(X[self.cols_]).fillna(self.medians_)
        if self.use_kf: Xc = KF_FN[self.dataset](Xc)
        A = Xc.values.astype(np.float32)
        return self.scaler_.transform(A).astype(np.float32) if self.scale else A

print("Maternal engineered features:", list(kf_maternal(DATA["maternal"]["Xd"].head()).columns))

In [ ]:
# %% [C5] FT-TRANSFORMER (PyTorch) — ordinal (CORAL) or softmax head =================
import torch, torch.nn as nn, torch.nn.functional as F
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE, "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

class FTNet(nn.Module):
    def __init__(self, n_feat, n_classes, d=32, n_layers=3, n_heads=4, drop=0.15, ordinal=True):
        super().__init__()
        self.ordinal, self.K = ordinal, n_classes
        self.W = nn.Parameter(torch.randn(n_feat, d) / math.sqrt(d))   # numerical feature tokenizer
        self.B = nn.Parameter(torch.zeros(n_feat, d))
        self.cls = nn.Parameter(torch.randn(1, 1, d) * 0.02)
        layer = nn.TransformerEncoderLayer(d, n_heads, dim_feedforward=2*d, dropout=drop,
                                           batch_first=True, norm_first=True, activation="gelu")
        self.enc = nn.TransformerEncoder(layer, n_layers, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(d)
        if ordinal:
            self.head = nn.Linear(d, 1, bias=False)                          # CORAL: shared weight
            self.ord_bias = nn.Parameter(torch.linspace(1.0, -1.0, n_classes - 1))  # K-1 thresholds
        else:
            self.head = nn.Linear(d, n_classes)
    def forward(self, x):
        tok = x.unsqueeze(-1) * self.W + self.B
        tok = torch.cat([self.cls.expand(x.size(0), -1, -1), tok], dim=1)
        z = self.norm(self.enc(tok)[:, 0])
        return self.head(z) + self.ord_bias if self.ordinal else self.head(z)

def ordinal_logits_to_probs(logits):
    cum = torch.sigmoid(logits)                                   # P(y > k)
    cum = torch.cummin(cum, dim=1).values                         # enforce rank consistency
    K1 = cum.size(1)
    p = [1 - cum[:, 0]] + [cum[:, k-1] - cum[:, k] for k in range(1, K1)] + [cum[:, -1]]
    p = torch.stack(p, 1).clamp_min(1e-6)
    return p / p.sum(1, keepdim=True)

def _loss(model, logits, y, w):
    if model.ordinal:
        levels = (y.unsqueeze(1) > torch.arange(model.K - 1, device=y.device).unsqueeze(0)).float()
        l = F.binary_cross_entropy_with_logits(logits, levels, reduction="none").mean(1)
    else:
        l = F.cross_entropy(logits, y, reduction="none")
    return (l * w).mean()

@torch.no_grad()
def ft_predict(model, X, bs=4096):
    model.eval(); out = []
    for i in range(0, len(X), bs):
        xb = torch.as_tensor(X[i:i+bs], dtype=torch.float32, device=DEVICE)
        lg = model(xb)
        out.append((ordinal_logits_to_probs(lg) if model.ordinal else F.softmax(lg, 1)).cpu().numpy())
    return np.concatenate(out)

def train_ft(Xtr, ytr, Xval, yval, K, ordinal=True, sw=None, seed=42, fixed_epochs=None):
    """Early stopping on (Xval, yval); or, with fixed_epochs, train exactly that many epochs (refit on all data)."""
    set_seed(seed); c = CFG["FT"]
    m = FTNet(Xtr.shape[1], K, c["d_token"], c["n_layers"], c["n_heads"], c["dropout"], ordinal).to(DEVICE)
    opt = torch.optim.AdamW(m.parameters(), lr=c["lr"], weight_decay=c["wd"])
    Xt = torch.as_tensor(Xtr, dtype=torch.float32, device=DEVICE)
    yt = torch.as_tensor(ytr, dtype=torch.long, device=DEVICE)
    wt = torch.as_tensor(np.ones(len(ytr)) if sw is None else sw, dtype=torch.float32, device=DEVICE)
    best, best_state, best_ep, wait = np.inf, None, 0, 0
    for ep in range(fixed_epochs or c["max_epochs"]):
        m.train(); perm = torch.randperm(len(Xt), device=DEVICE)
        for i in range(0, len(Xt), c["batch"]):
            b = perm[i:i+c["batch"]]
            if len(b) < 2: continue
            opt.zero_grad(); loss = _loss(m, m(Xt[b]), yt[b], wt[b]); loss.backward()
            nn.utils.clip_grad_norm_(m.parameters(), 1.0); opt.step()
        if fixed_epochs: continue
        v = log_loss(yval, ft_predict(m, Xval), labels=list(range(K)))
        if v < best - 1e-4: best, best_state, best_ep, wait = v, copy.deepcopy(m.state_dict()), ep, 0
        else:
            wait += 1
            if wait >= c["patience"]: break
    if fixed_epochs: m.best_epoch_ = fixed_epochs - 1
    else: m.load_state_dict(best_state); m.best_epoch_ = best_ep
    m.n_params_ = sum(p.numel() for p in m.parameters())
    return m

class FTClassifier(BaseEstimator, ClassifierMixin):
    """Plain FT-Transformer baseline (softmax, no knowledge features, no fusion)."""
    def __init__(self, ordinal=False, val_frac=0.15, seed=42):
        self.ordinal, self.val_frac, self.seed = ordinal, val_frac, seed
    def fit(self, X, y):
        y = np.asarray(y); self.classes_ = np.unique(y)
        tr, va = next(StratifiedShuffleSplit(1, test_size=self.val_frac, random_state=self.seed).split(X, y))
        self.model_ = train_ft(X[tr], y[tr], X[va], y[va], len(self.classes_), self.ordinal, None, self.seed)
        return self
    def predict_proba(self, X): return ft_predict(self.model_, np.asarray(X, dtype=np.float32))
    def predict(self, X): return self.predict_proba(X).argmax(1)

In [ ]:
# %% [C6] BASELINE MODELS =============================================================
class CatWrap(BaseEstimator, ClassifierMixin):
    """Thin sklearn wrapper (avoids CatBoost/sklearn tag issues in Pipeline/Stacking)."""
    def __init__(self, iterations=800, depth=6, learning_rate=0.05, random_state=42):
        self.iterations, self.depth, self.learning_rate, self.random_state = iterations, depth, learning_rate, random_state
    def fit(self, X, y, sample_weight=None):
        self.classes_ = np.unique(y)
        self.model_ = CatBoostClassifier(iterations=self.iterations, depth=self.depth, learning_rate=self.learning_rate,
                                         loss_function="MultiClass", random_seed=self.random_state, verbose=0,
                                         allow_writing_files=False, thread_count=-1)
        self.model_.fit(X, y, sample_weight=sample_weight); return self
    def predict_proba(self, X): return self.model_.predict_proba(X)
    def predict(self, X): return self.classes_[self.predict_proba(X).argmax(1)]

def make_baselines(ds):
    S = CFG["SEED"]
    xgb = lambda: XGBClassifier(n_estimators=400, max_depth=6, learning_rate=0.05, subsample=0.9, colsample_bytree=0.9,
                                objective="multi:softprob", eval_metric="mlogloss", random_state=S, n_jobs=-1, verbosity=0)
    P = lambda clf, kf=False: Pipeline([("prep", ClinicalPreprocessor(ds, use_kf=kf, scale=True)), ("clf", clf)])
    return {
        "Logistic Regression": lambda: P(LogisticRegression(max_iter=3000, C=1.0)),
        "KNN":                 lambda: P(KNeighborsClassifier(n_neighbors=7, weights="distance")),
        "SVM (RBF)":           lambda: P(SVC(C=10, gamma="scale", probability=True, random_state=S)),
        "Decision Tree":       lambda: P(DecisionTreeClassifier(random_state=S)),
        "Naive Bayes":         lambda: P(GaussianNB()),
        "Random Forest":       lambda: P(RandomForestClassifier(n_estimators=500, random_state=S, n_jobs=-1)),
        "Extra Trees":         lambda: P(ExtraTreesClassifier(n_estimators=500, random_state=S, n_jobs=-1)),
        "AdaBoost":            lambda: P(AdaBoostClassifier(n_estimators=300, learning_rate=0.5, random_state=S)),
        "Gradient Boosting":   lambda: P(GradientBoostingClassifier(n_estimators=300, learning_rate=0.05, max_depth=3, random_state=S)),
        "XGBoost":             lambda: P(xgb()),
        "LightGBM":            lambda: P(LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=31, random_state=S, verbose=-1, n_jobs=-1)),
        "CatBoost":            lambda: P(CatWrap(random_state=S)),
        "MLP":                 lambda: P(MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=2000, early_stopping=True, random_state=S)),
        "FT-Transformer":      lambda: P(FTClassifier(ordinal=False, seed=S)),
        "Stacking (Lit.)":     lambda: P(StackingClassifier(
                                    estimators=[("rf", RandomForestClassifier(n_estimators=300, random_state=S, n_jobs=-1)),
                                                ("xgb", xgb()),
                                                ("gb", GradientBoostingClassifier(n_estimators=200, random_state=S))],
                                    final_estimator=xgb(), cv=StratifiedKFold(5, shuffle=True, random_state=S),
                                    stack_method="predict_proba", n_jobs=1)),
        "CatBoost + KF":       lambda: P(CatWrap(random_state=S), kf=True),
    }

HYPERPARAMS = pd.DataFrame([
    ("Logistic Regression", "C=1.0, max_iter=3000, L2"), ("KNN", "k=7, distance weighting"),
    ("SVM (RBF)", "C=10, gamma=scale, Platt probabilities"), ("Decision Tree", "unpruned, gini"),
    ("Naive Bayes", "Gaussian"), ("Random Forest", "500 trees"), ("Extra Trees", "500 trees"),
    ("AdaBoost", "300 estimators, lr=0.5"), ("Gradient Boosting", "300 est., lr=0.05, depth=3"),
    ("XGBoost", "400 est., depth=6, lr=0.05, subsample=0.9, colsample=0.9"),
    ("LightGBM", "400 est., lr=0.05, num_leaves=31"), ("CatBoost", "800 it., depth=6, lr=0.05"),
    ("MLP", "128-64, ReLU, Adam, early stopping"),
    ("FT-Transformer", f"d={CFG['FT']['d_token']}, layers={CFG['FT']['n_layers']}, heads={CFG['FT']['n_heads']}, dropout={CFG['FT']['dropout']}, AdamW lr={CFG['FT']['lr']}, early stop (patience {CFG['FT']['patience']})"),
    ("Stacking (Lit.)", "RF(300)+XGB+GB(200) -> XGB meta, 5-fold inner CV"),
    ("CatBoost + KF", "CatBoost + clinical knowledge features"),
    (PROP, f"Branches: FT (ordinal CORAL) + CatBoost (early-stopped, then refit) + GB(300, lr 0.05, depth 3); inner {CFG['INNER_FOLDS']} group-CV OOF stacking -> multinomial LR meta (C=1); cross-conformal (LAC + Mondrian); high-class cost x{CFG['HIGH_COST']}"),
], columns=["Model", "Hyperparameters"])
save_table(HYPERPARAMS, "T03_hyperparameters")

In [ ]:
# %% [C7] PROPOSED MODEL v2: OCH-MRNet (out-of-fold stacking + cross-conformal) ================
def nll(p, y): return -np.mean(np.log(p[np.arange(len(y)), y] + 1e-12))

def make_weights(y, K, high_cost, balanced=True):
    if not balanced: return np.ones(len(y))
    cnt = np.bincount(y, minlength=K).astype(float); w = len(y) / (K * np.maximum(cnt, 1))
    w[K - 1] *= high_cost; sw = w[y]
    return sw / sw.mean()

def conformal_q(scores, alpha):
    n = len(scores)
    if n == 0 or math.ceil((n + 1) * (1 - alpha)) > n: return 1.0
    return float(np.quantile(scores, math.ceil((n + 1) * (1 - alpha)) / n, method="higher"))

META_SETS = {"full": ["ft", "cat", "gb"], "tree_only": ["cat", "gb"], "no_gb": ["ft", "cat"]}
def meta_X(P, keys): return np.hstack([np.log(np.clip(P[k], 1e-6, 1)) for k in keys])
def new_meta(): return LogisticRegression(max_iter=5000, C=1.0)

class OCHMRNet:
    """Knowledge features -> three heterogeneous branches [FT-Transformer (ordinal CORAL), CatBoost, Gradient Boosting]
       trained with inner group-CV -> cost-sensitive out-of-fold stacking meta-learner -> branches refit on the
       WHOLE training fold (no data held out) -> cross-conformal prediction sets (marginal + class-conditional)."""
    def __init__(self, dataset, K, use_kf=True, ordinal=True, cost=True, seed=42):
        self.dataset, self.K, self.use_kf, self.ordinal, self.cost, self.seed = dataset, K, use_kf, ordinal, cost, seed

    def _fit_branches(self, Za, ya, sw, Zv=None, yv=None):
        ft = train_ft(Za, ya, Zv, yv, self.K, self.ordinal, sw, self.seed,
                      fixed_epochs=None if Zv is not None else self.ft_epochs_)
        kw = dict(depth=6, learning_rate=0.03, loss_function="MultiClass", random_seed=self.seed, verbose=0,
                  allow_writing_files=False, thread_count=-1)
        if Zv is not None:
            cat = CatBoostClassifier(iterations=2000, early_stopping_rounds=100, **kw)
            cat.fit(Za, ya, sample_weight=sw, eval_set=(Zv, yv))
        else:
            cat = CatBoostClassifier(iterations=self.cat_iters_, **kw); cat.fit(Za, ya, sample_weight=sw)
        gb = GradientBoostingClassifier(n_estimators=300, learning_rate=0.05, max_depth=3,
                                        random_state=self.seed).fit(Za, ya, sample_weight=sw)
        return dict(ft=ft, cat=cat, gb=gb)

    def _probs(self, M, Z):
        return dict(ft=ft_predict(M["ft"], Z), cat=M["cat"].predict_proba(Z), gb=M["gb"].predict_proba(Z))

    def fit(self, X, y, groups):
        y = np.asarray(y); groups = np.asarray(groups); t0 = time.time()
        self.prep_ = ClinicalPreprocessor(self.dataset, self.use_kf, True).fit(X)
        Z = self.prep_.transform(X)
        sw = make_weights(y, self.K, CFG["HIGH_COST"], balanced=self.cost)
        # 1) inner group-CV -> out-of-fold branch probabilities for EVERY training row
        oof = {k: np.zeros((len(y), self.K)) for k in ["ft", "cat", "gb"]}; eps, its = [], []
        inner = StratifiedGroupKFold(CFG["INNER_FOLDS"][self.dataset], shuffle=True, random_state=self.seed)
        for a, v in inner.split(Z, y, groups):
            M = self._fit_branches(Z[a], y[a], sw[a], Z[v], y[v]); P = self._probs(M, Z[v])
            for k in oof: oof[k][v] = P[k]
            eps.append(M["ft"].best_epoch_ + 1); its.append(max(50, M["cat"].get_best_iteration() + 1))
        self.ft_epochs_, self.cat_iters_ = int(np.median(eps)), int(np.median(its))
        # 2) class-balanced stacking meta-learners on OOF log-probabilities
        msw = make_weights(y, self.K, 1.0, balanced=self.cost)          # meta: class-balanced only (cost already in branches)
        self.metas_ = {n: new_meta().fit(meta_X(oof, ks), y, sample_weight=msw) for n, ks in META_SETS.items()}
        # 3) cross-conformal scores: meta re-fitted by cross-fitting so every score is out-of-sample
        Pcv = np.zeros((len(y), self.K)); F = meta_X(oof, META_SETS["full"])
        for a, v in StratifiedGroupKFold(5, shuffle=True, random_state=self.seed + 7).split(F, y, groups):
            Pcv[v] = new_meta().fit(F[a], y[a], sample_weight=msw[a]).predict_proba(F[v])
        self.cal_scores_ = 1 - Pcv[np.arange(len(y)), y]; self.cal_y_ = y
        # 4) refit all branches on the whole training fold
        self.M_ = self._fit_branches(Z, y, sw)
        self.fit_time_ = time.time() - t0
        return self

    def variants(self, X):
        Z = self.prep_.transform(X if isinstance(X, pd.DataFrame) else pd.DataFrame(X, columns=self.prep_.cols_))
        P = self._probs(self.M_, Z)
        out = {n: self.metas_[n].predict_proba(meta_X(P, ks)) for n, ks in META_SETS.items()}
        out["avg"] = (P["ft"] + P["cat"] + P["gb"]) / 3; out["ft_only"] = P["ft"]
        return out

    def predict_proba(self, X): return self.variants(X)["full"]

    def predict_sets(self, P, alpha, mondrian=False):
        if not mondrian:
            return (1 - P) <= conformal_q(self.cal_scores_, alpha)
        qs = np.array([conformal_q(self.cal_scores_[self.cal_y_ == k], alpha) for k in range(self.K)])
        return (1 - P) <= qs[None, :]

    def n_params(self):
        return int(self.M_["ft"].n_params_), int(self.M_["cat"].tree_count_)

print("OCH-MRNet v2 defined.")

In [ ]:
# %% [C8] METRICS =====================================================================
def ece_score(y, P, n_bins=15):
    conf, acc = P.max(1), (P.argmax(1) == y).astype(float)
    bins, e = np.linspace(0, 1, n_bins + 1), 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any(): e += m.mean() * abs(acc[m].mean() - conf[m].mean())
    return e

def compute_metrics(y, P, K, high):
    y = np.asarray(y); pred = P.argmax(1)
    p, r, f, _ = precision_recall_fscore_support(y, pred, labels=list(range(K)), average="macro", zero_division=0)
    pc, rc, _, _ = precision_recall_fscore_support(y, pred, labels=list(range(K)), average=None, zero_division=0)
    try: aucv = roc_auc_score(y, P, multi_class="ovr", average="macro", labels=list(range(K)))
    except Exception: aucv = np.nan
    Y = np.eye(K)[y]
    return dict(Accuracy=accuracy_score(y, pred), BalancedAcc=balanced_accuracy_score(y, pred),
                MacroP=p, MacroR=r, MacroF1=f, MCC=matthews_corrcoef(y, pred),
                Kappa=cohen_kappa_score(y, pred), QWK=cohen_kappa_score(y, pred, weights="quadratic"),
                AUC=aucv, HighRecall=rc[high], HighPrecision=pc[high],
                UnderTriage=float(np.mean(pred < y)),               # severity under-estimation rate
                ECE=ece_score(y, P), Brier=float(np.mean(np.sum((P - Y) ** 2, 1))),
                LogLoss=log_loss(y, np.clip(P, 1e-12, 1), labels=list(range(K))))

METRIC_COLS = ["Accuracy", "BalancedAcc", "MacroP", "MacroR", "MacroF1", "MCC", "Kappa", "QWK", "AUC",
               "HighRecall", "HighPrecision", "UnderTriage", "ECE", "Brier", "LogLoss"]
print("Metrics ready.")

In [ ]:
# %% [C9] MAIN EXPERIMENT ENGINE (Protocol A = naive raw split, Protocol B = dedup + group CV) =====
ROBUST_MODELS = ["Random Forest", "XGBoost", "CatBoost", "Stacking (Lit.)", "FT-Transformer", PROP]
ABL_MAP = {"full": "OCH-MRNet (Full)", "avg": "w/o Stacking meta-learner (avg)", "tree_only": "w/o FT branch (CatBoost+GB)",
           "no_gb": "w/o GB branch (FT+CatBoost)", "ft_only": "w/o Tree branches (FT only)"}

def overlap_rate(Xtr, ytr, Xte, yte):
    a = pd.util.hash_pandas_object(Xtr.assign(__y=ytr), index=False)
    b = pd.util.hash_pandas_object(Xte.assign(__y=yte), index=False)
    return float(b.isin(set(a)).mean())

def add_noise(X, std, level, rng):
    if level == 0: return X
    Xn = X.copy()
    for c in X.columns:
        if X[c].nunique() > 2: Xn[c] = X[c] + rng.normal(0, level * std[c], len(X))
    return Xn

def run_protocol(key, protocol):
    D = DATA[key]; K, high = D["K"], D["high"]
    if protocol == "B": X, y, groups, n_rep = D["Xd"], D["yd"], D["gd"], CFG["N_REPEATS"][key]
    else:               X, y, n_rep = D["X"], D["y"], CFG["N_REPEATS_A"]; groups = np.arange(len(y))
    baselines = make_baselines(key)
    res, conf_rows, rob_rows, oof, set_log = [], [], [], {}, []
    complexity = {}
    total = n_rep * CFG["N_SPLITS"]; t_start = time.time()
    for r in range(n_rep):
        skf = StratifiedGroupKFold(n_splits=CFG["N_SPLITS"], shuffle=True, random_state=CFG["SEED"] + r)
        for f, (tr, te) in enumerate(skf.split(np.zeros(len(y)), y, groups)):
            Xtr, Xte, ytr, yte = X.iloc[tr], X.iloc[te], y[tr], y[te]
            ov = overlap_rate(Xtr, ytr, Xte, yte)
            std = Xtr.std(); rng_seed = CFG["SEED"] + 1000 * r + f
            meta = dict(Dataset=D["name"], Protocol=protocol, Repeat=r, Fold=f, TrainTestOverlap=ov)

            def record(name, P, fit_t, inf_t, extra=None):
                row = {**meta, "Model": name, **compute_metrics(yte, P, K, high), "FitTime_s": fit_t,
                       "InferMs_per_sample": 1000 * inf_t / len(yte)}
                if extra: row.update(extra)
                res.append(row)
                if r == 0:
                    oof.setdefault(name, np.full((len(y), K), np.nan))[te] = P

            # ---- baselines
            for name, factory in baselines.items():
                set_seed(CFG["SEED"]); m = factory(); t0 = time.time(); m.fit(Xtr, ytr); ft_ = time.time() - t0
                t0 = time.time(); P = m.predict_proba(Xte); it_ = time.time() - t0
                record(name, P, ft_, it_)
                if r == 0 and protocol == "B" and name in ROBUST_MODELS:
                    for lv in CFG["NOISE_LEVELS"]:
                        Pn = m.predict_proba(add_noise(Xte, std, lv, np.random.default_rng(rng_seed)))
                        rob_rows.append(dict(Dataset=D["name"], Model=name, Fold=f, Noise=lv,
                                             MacroF1=f1_score(yte, Pn.argmax(1), average="macro"),
                                             HighRecall=recall_score(yte, Pn.argmax(1), labels=[high], average="macro", zero_division=0)))
            # ---- proposed
            set_seed(CFG["SEED"]); m = OCHMRNet(key, K, seed=CFG["SEED"] + f).fit(Xtr, ytr, groups[tr])
            t0 = time.time(); V = m.variants(Xte); it_ = time.time() - t0
            record(PROP, V["full"], m.fit_time_, it_)
            if r == 0:
                oof.setdefault("OCH-MRNet (avg fusion)", np.full((len(y), K), np.nan))[te] = V["avg"]
            complexity.setdefault("ft_params", []).append(m.n_params()[0]); complexity.setdefault("cat_trees", []).append(m.n_params()[1])
            if protocol == "B":
                for vk, vname in ABL_MAP.items():
                    res.append({**meta, "Model": vname, "Ablation": True, **compute_metrics(yte, V[vk], K, high)})
                if CFG["RUN_ABLATION"] and r < CFG["ABL_REPEATS"][key]:
                    for vname, kw in [("w/o Knowledge features", dict(use_kf=False)),
                                      ("w/o Ordinal loss (CE)", dict(ordinal=False)),
                                      ("w/o Cost-sensitive weighting", dict(cost=False))]:
                        set_seed(CFG["SEED"]); ma = OCHMRNet(key, K, seed=CFG["SEED"] + f, **kw).fit(Xtr, ytr, groups[tr])
                        res.append({**meta, "Model": vname, "Ablation": True, **compute_metrics(yte, ma.predict_proba(Xte), K, high)})
                # conformal prediction
                for alpha in CFG["CONF_ALPHAS"]:
                    for mond in [False, True]:
                        S = m.predict_sets(V["full"], alpha, mond); sz = S.sum(1)
                        cov = S[np.arange(len(yte)), yte]
                        conf_rows.append(dict(Dataset=D["name"], Repeat=r, Fold=f, Alpha=alpha,
                                              Method="Class-conditional (Mondrian)" if mond else "Marginal (LAC)",
                                              Coverage=cov.mean(), HighClassCoverage=cov[yte == high].mean() if (yte == high).any() else np.nan,
                                              AvgSetSize=sz.mean(), Singleton=(sz == 1).mean(), Empty=(sz == 0).mean(),
                                              Abstain=(sz != 1).mean(),
                                              SingletonAcc=(cov[sz == 1].mean() if (sz == 1).any() else np.nan)))
                        if r == 0 and alpha == CFG["MAIN_ALPHA"] and mond:
                            for i in range(len(yte)):
                                set_log.append(dict(Dataset=D["name"], TrueClass=D["classes"][yte[i]],
                                                    PredSet="{" + ",".join(D["classes"][k] for k in np.where(S[i])[0]) + "}"))
                # robustness for proposed
                if r == 0:
                    for lv in CFG["NOISE_LEVELS"]:
                        Pn = m.predict_proba(add_noise(Xte, std, lv, np.random.default_rng(rng_seed)))
                        rob_rows.append(dict(Dataset=D["name"], Model=PROP, Fold=f, Noise=lv,
                                             MacroF1=f1_score(yte, Pn.argmax(1), average="macro"),
                                             HighRecall=recall_score(yte, Pn.argmax(1), labels=[high], average="macro", zero_division=0)))
            done = r * CFG["N_SPLITS"] + f + 1; el = time.time() - t_start
            last = pd.DataFrame(res)
            last = last[(last.Repeat == r) & (last.Fold == f) & (last.Model == PROP)].iloc[0]
            print(f"[{key}|{protocol}] fold {done}/{total} | overlap={ov:.2%} | {PROP} F1={last.MacroF1:.4f} "
                  f"HighRec={last.HighRecall:.4f} | elapsed {el/60:.1f} min, ETA {el/done*(total-done)/60:.1f} min")
    out = dict(res=pd.DataFrame(res), conf=pd.DataFrame(conf_rows), rob=pd.DataFrame(rob_rows), oof=oof,
               y=y, X=X, sets=pd.DataFrame(set_log), complexity=complexity)
    out["res"].to_csv(os.path.join(TAB, f"raw_results_{key}_{protocol}.csv"), index=False)
    return out

print("Engine ready.")

In [ ]:
# %% [C10] RUN EXPERIMENTS (long cell — total ~1.5–2.5 h on T4; set FAST_MODE=True for a dry run) ====
RUNS = {}
RUNS[("maternal", "B")] = run_protocol("maternal", "B")
RUNS[("maternal", "A")] = run_protocol("maternal", "A")      # leakage study
if CFG["RUN_FETAL"]:
    RUNS[("fetal", "B")] = run_protocol("fetal", "B")
print("ALL EXPERIMENTS DONE")

In [ ]:
# %% [C11] RESULT TABLES (main, leakage, ablation, calibration, conformal, complexity) ==================
def mean_std_table(df, models=None, cols=METRIC_COLS, digits=4):
    g = df.groupby("Model")[cols]; m, s = g.mean(), g.std()
    out = pd.DataFrame({c: [f"{m.loc[i, c]:.{digits}f} ± {s.loc[i, c]:.{digits}f}" for i in m.index] for c in cols}, index=m.index)
    out["_sort"] = m["MacroF1"]
    if models is not None: out = out.loc[[x for x in models if x in out.index]]
    return out.sort_values("_sort", ascending=False).drop(columns="_sort").reset_index()

MAIN_TABLES, MEANS = {}, {}
for (key, prot), R in RUNS.items():
    if prot != "B": continue
    df = R["res"][R["res"].get("Ablation", pd.Series(False, index=R["res"].index)).fillna(False) == False]
    tbl = mean_std_table(df); MAIN_TABLES[key] = tbl
    MEANS[key] = df.groupby("Model")[METRIC_COLS].mean()
    save_table(tbl, f"T05_main_results_{key}")
    print(f"\n==== MAIN RESULTS ({key}, Protocol B) ====\n", tbl[["Model", "Accuracy", "MacroF1", "MCC", "QWK", "AUC", "HighRecall", "ECE"]].to_string(index=False))

# ---- Leakage table (maternal: Protocol A vs B)
A = RUNS[("maternal", "A")]["res"]; B = RUNS[("maternal", "B")]["res"]
B = B[B.get("Ablation", pd.Series(False, index=B.index)).fillna(False) == False]
leak = pd.DataFrame({
    "Acc_A (naive)": A.groupby("Model").Accuracy.mean(), "Acc_B (dedup+group)": B.groupby("Model").Accuracy.mean(),
    "F1_A (naive)": A.groupby("Model").MacroF1.mean(), "F1_B (dedup+group)": B.groupby("Model").MacroF1.mean()})
leak["ΔAcc (inflation, pp)"] = (leak["Acc_A (naive)"] - leak["Acc_B (dedup+group)"]) * 100
leak["ΔF1 (inflation, pp)"] = (leak["F1_A (naive)"] - leak["F1_B (dedup+group)"]) * 100
leak = leak.sort_values("F1_A (naive)", ascending=False).round(4).reset_index()
save_table(leak, "T04_leakage_protocolA_vs_B")
print("\nMean train-test exact overlap — Protocol A: {:.2%} | Protocol B: {:.2%}".format(A.TrainTestOverlap.mean(), B.TrainTestOverlap.mean()))
print(leak.to_string(index=False))

# ---- Ablation
ABL_ORDER = list(ABL_MAP.values()) + ["w/o Knowledge features", "w/o Ordinal loss (CE)", "w/o Cost-sensitive weighting"]
ABL_TABLES = {}
for (key, prot), R in RUNS.items():
    if prot != "B": continue
    ab = R["res"][R["res"].Model.isin(ABL_ORDER) & (R["res"].Repeat < CFG["ABL_REPEATS"][key])]   # same folds for every variant
    t = ab.groupby("Model")[["Accuracy", "MacroF1", "MCC", "QWK", "HighRecall", "UnderTriage", "ECE", "Brier"]].agg(["mean", "std"])
    t.columns = [f"{a}_{b}" for a, b in t.columns]; t = t.loc[[m for m in ABL_ORDER if m in t.index]]
    full = t.loc["OCH-MRNet (Full)"]
    t["ΔMacroF1 vs Full (pp)"] = (t["MacroF1_mean"] - full["MacroF1_mean"]) * 100
    t["ΔHighRecall vs Full (pp)"] = (t["HighRecall_mean"] - full["HighRecall_mean"]) * 100
    ABL_TABLES[key] = t.round(4).reset_index()
    save_table(ABL_TABLES[key], f"T06_ablation_{key}")
    print(f"\n==== ABLATION ({key}) ====\n", ABL_TABLES[key][["Model", "MacroF1_mean", "HighRecall_mean", "ECE_mean", "ΔMacroF1 vs Full (pp)"]].to_string(index=False))

# ---- Calibration + conformal
for (key, prot), R in RUNS.items():
    if prot != "B": continue
    cal = MEANS[key][["ECE", "Brier", "LogLoss"]].copy()
    cal.loc["OCH-MRNet (avg fusion, no meta-learner)"] = R["res"][R["res"].Model == "w/o Stacking meta-learner (avg)"][["ECE", "Brier", "LogLoss"]].mean()
    save_table(cal.sort_values("ECE").round(4).reset_index(), f"T07a_calibration_{key}")
    cf = R["conf"].groupby(["Method", "Alpha"])[["Coverage", "HighClassCoverage", "AvgSetSize", "Singleton", "Empty", "Abstain", "SingletonAcc"]].mean().reset_index()
    cf.insert(2, "Target coverage", 1 - cf.Alpha)
    save_table(cf.round(4), f"T07b_conformal_{key}")
    print(f"\n==== CONFORMAL ({key}) ====\n", cf.round(3).to_string(index=False))
    if len(R["sets"]):
        tri = pd.crosstab(R["sets"].PredSet, R["sets"].TrueClass)
        save_table(tri.reset_index(), f"T07c_triage_sets_{key}")

# ---- Complexity
comp_rows = []
for (key, prot), R in RUNS.items():
    if prot != "B": continue
    df = R["res"][R["res"].get("Ablation", pd.Series(False, index=R["res"].index)).fillna(False) == False]
    c = df.groupby("Model")[["FitTime_s", "InferMs_per_sample"]].mean().round(4)
    c.insert(0, "Dataset", DATA[key]["name"])
    c["Params / trees"] = ""
    c.loc[PROP, "Params / trees"] = f"FT {int(np.mean(R['complexity']['ft_params'])):,} params + CatBoost ~{int(np.mean(R['complexity']['cat_trees']))} trees"
    comp_rows.append(c.reset_index())
save_table(pd.concat(comp_rows), "T10_computational_cost")

In [ ]:
# %% [C12] STATISTICAL TESTS (Friedman + Nemenyi CD, Wilcoxon & Nadeau–Bengio corrected t-test, Holm) ====
Q_NEMENYI = {2: 1.960, 3: 2.343, 4: 2.569, 5: 2.728, 6: 2.850, 7: 2.949, 8: 3.031, 9: 3.102, 10: 3.164, 11: 3.219,
             12: 3.268, 13: 3.313, 14: 3.354, 15: 3.391, 16: 3.426, 17: 3.458, 18: 3.489, 19: 3.517, 20: 3.544}

def holm(p):
    p = np.asarray(p); o = np.argsort(p); adj = np.empty_like(p); run = 0
    for i, idx in enumerate(o):
        run = max(run, (len(p) - i) * p[idx]); adj[idx] = min(1, run)
    return adj

STATS = {}
for (key, prot), R in RUNS.items():
    if prot != "B": continue
    df = R["res"][R["res"].get("Ablation", pd.Series(False, index=R["res"].index)).fillna(False) == False]
    M = df.pivot_table(index=["Repeat", "Fold"], columns="Model", values="MacroF1")
    fr = stats.friedmanchisquare(*[M[c].values for c in M.columns])
    ranks = M.rank(axis=1, ascending=False).mean().sort_values()
    k, N = M.shape[1], M.shape[0]; CD = Q_NEMENYI.get(k, 3.6) * math.sqrt(k * (k + 1) / (6 * N))
    n_te = len(R["y"]) / CFG["N_SPLITS"]; n_tr = len(R["y"]) - n_te
    rows = []
    for c in M.columns:
        if c == PROP: continue
        d = M[PROP].values - M[c].values
        try: pw = stats.wilcoxon(M[PROP].values, M[c].values, zero_method="zsplit").pvalue
        except Exception: pw = 1.0
        var = d.var(ddof=1); J = len(d)
        tcorr = d.mean() / math.sqrt((1 / J + n_te / n_tr) * var) if var > 0 else 0.0
        pnb = 2 * stats.t.sf(abs(tcorr), J - 1)
        rows.append(dict(Comparison=f"{PROP} vs {c}", MeanΔMacroF1=d.mean(), Wilcoxon_p=pw, NB_corrected_t=tcorr, NB_p=pnb,
                         RankDiff=ranks[c] - ranks[PROP], Nemenyi_significant=abs(ranks[c] - ranks[PROP]) > CD))
    st = pd.DataFrame(rows); st["Wilcoxon_p_Holm"] = holm(st.Wilcoxon_p); st["NB_p_Holm"] = holm(st.NB_p)
    st = st.sort_values("MeanΔMacroF1").round(5)
    save_table(st, f"T08_statistical_tests_{key}")
    STATS[key] = dict(friedman=fr, ranks=ranks, CD=CD, N=N, k=k)
    print(f"\n[{key}] Friedman χ²={fr.statistic:.3f}, p={fr.pvalue:.3e} | k={k}, N={N}, Nemenyi CD={CD:.3f}")
    print(st[["Comparison", "MeanΔMacroF1", "Wilcoxon_p_Holm", "NB_p_Holm", "Nemenyi_significant"]].to_string(index=False))
    pd.DataFrame({"Model": ranks.index, "AvgRank": ranks.values}).to_csv(os.path.join(TAB, f"T08b_avg_ranks_{key}.csv"), index=False)

In [ ]:
# %% [C13] FIGURES: EDA, leakage, comparison, confusion, ROC/PR, reliability, conformal, CD, DCA, robustness, ablation, fairness ===
# Fig 2 — class distribution raw vs dedup
fig, axes = plt.subplots(1, len(DATA), figsize=(6 * len(DATA), 4.2))
axes = np.atleast_1d(axes)
for ax, (key, D) in zip(axes, DATA.items()):
    dd = pd.DataFrame({"Class": D["classes"] * 2,
                       "Count": [D["audit"][f"Raw_{c}"] for c in D["classes"]] + [D["audit"][f"Dedup_{c}"] for c in D["classes"]],
                       "Version": ["Raw"] * D["K"] + ["Deduplicated"] * D["K"]})
    sns.barplot(data=dd, x="Class", y="Count", hue="Version", ax=ax, palette=["#95A5A6", "#2E86C1"])
    for p in ax.patches:
        if p.get_height() > 0: ax.annotate(int(p.get_height()), (p.get_x() + p.get_width() / 2, p.get_height()), ha="center", va="bottom", fontsize=9)
    ax.set_title(f"{D['name']}\n{D['audit']['Exact_duplicates']} duplicates ({D['audit']['Duplicate_pct']}%)")
savefig(fig, "Fig02_class_distribution_raw_vs_dedup")

# Fig 3 — maternal EDA
Dm = DATA["maternal"]; E = Dm["Xd"].copy(); E["Risk"] = pd.Categorical([Dm["classes"][i] for i in Dm["yd"]], Dm["classes"])
fig, ax = plt.subplots(figsize=(7, 5.8))
sns.heatmap(Dm["Xd"].assign(RiskLevel=Dm["yd"]).corr(method="spearman"), annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=ax, square=True)
ax.set_title("Spearman correlation (deduplicated maternal data)"); savefig(fig, "Fig03a_maternal_correlation")
fig, axes = plt.subplots(2, 3, figsize=(14, 7.5))
for ax, c in zip(axes.ravel(), Dm["Xd"].columns):
    sns.violinplot(data=E, x="Risk", y=c, ax=ax, palette=["#27AE60", "#F39C12", "#C0392B"], cut=0, inner="quartile"); ax.set_xlabel("")
savefig(fig, "Fig03b_maternal_feature_by_risk")
if "fetal" in DATA:
    Df = DATA["fetal"]; cc = Df["Xd"].assign(fetal_health=Df["yd"]).corr(method="spearman")["fetal_health"].drop("fetal_health").sort_values()
    fig, ax = plt.subplots(figsize=(7, 7)); cc.plot.barh(ax=ax, color=np.where(cc > 0, "#C0392B", "#2E86C1"))
    ax.set_title("Fetal CTG: Spearman correlation with fetal state"); savefig(fig, "Fig03c_fetal_feature_correlation")

# Fig 4 — leakage
lk = leak.set_index("Model")
order = lk.sort_values("F1_B (dedup+group)", ascending=False).index
fig, ax = plt.subplots(figsize=(13, 5))
xx = np.arange(len(order)); w = 0.38
ax.bar(xx - w / 2, lk.loc[order, "F1_A (naive)"], w, label=f"Protocol A: naive random split (overlap {A.TrainTestOverlap.mean():.0%})", color="#E59866")
ax.bar(xx + w / 2, lk.loc[order, "F1_B (dedup+group)"], w, label="Protocol B: dedup + group-stratified CV (overlap 0%)", color="#2874A6")
for i, m in enumerate(order): ax.text(i, lk.loc[m, "F1_A (naive)"] + 0.01, f"+{lk.loc[m, 'ΔF1 (inflation, pp)']:.1f}", ha="center", fontsize=8, rotation=90)
ax.set_xticks(xx); ax.set_xticklabels(order, rotation=40, ha="right"); ax.set_ylabel("Macro-F1"); ax.set_ylim(0.4, 1.08)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.32), ncol=2, frameon=False)
ax.set_title("Performance inflation caused by duplicate leakage (Maternal Health Risk)")
savefig(fig, "Fig04_leakage_inflation")

# Fig 5 — main comparison
for key in MEANS:
    R = RUNS[(key, "B")]; df = R["res"][R["res"].get("Ablation", pd.Series(False, index=R["res"].index)).fillna(False) == False]
    g = df.groupby("Model")[["MacroF1", "HighRecall"]].agg(["mean", "std"]).sort_values(("MacroF1", "mean"))
    fig, axes = plt.subplots(1, 2, figsize=(13, 6.5), sharey=True)
    for ax, met in zip(axes, ["MacroF1", "HighRecall"]):
        cols = [PROP_COLOR if i == PROP else BASE_COLOR for i in g.index]
        ax.barh(g.index, g[(met, "mean")], xerr=g[(met, "std")], color=cols, capsize=3)
        lo = max(0, g[(met, "mean")].min() - 0.1); ax.set_xlim(lo, 1.0)
        ax.set_xlabel(f"{met} (mean ± SD over {len(df[['Repeat','Fold']].drop_duplicates())} folds)")
        ax.set_title({"MacroF1": "Macro-F1", "HighRecall": f"Recall of '{DATA[key]['classes'][-1]}' class"}[met])
    fig.suptitle(f"{DATA[key]['name']} — leakage-free comparison", fontweight="bold")
    savefig(fig, f"Fig05_model_comparison_{key}")

# helpers for OOF (repeat 0, each sample predicted once)
def best_baseline(key):
    m = MEANS[key].drop(index=[PROP], errors="ignore"); return m.MacroF1.idxmax()

# Fig 6 — confusion matrices
for key in MEANS:
    R, D = RUNS[(key, "B")], DATA[key]
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
    for ax, name in zip(axes, [PROP, best_baseline(key)]):
        P = R["oof"][name]; cm = confusion_matrix(R["y"], P.argmax(1)); cmn = cm / cm.sum(1, keepdims=True)
        lab = np.array([[f"{cm[i, j]}\n({cmn[i, j]:.1%})" for j in range(D["K"])] for i in range(D["K"])])
        sns.heatmap(cmn, annot=lab, fmt="", cmap="Blues", ax=ax, xticklabels=D["classes"], yticklabels=D["classes"], cbar=False)
        ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(name)
    savefig(fig, f"Fig06_confusion_{key}")

# Fig 7 — ROC + PR (proposed)
for key in MEANS:
    R, D = RUNS[(key, "B")], DATA[key]; P = R["oof"][PROP]; Y = label_binarize(R["y"], classes=list(range(D["K"])))
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.6)); colors = ["#27AE60", "#F39C12", "#C0392B"]
    for k in range(D["K"]):
        fpr, tpr, _ = roc_curve(Y[:, k], P[:, k]); axes[0].plot(fpr, tpr, color=colors[k], lw=2, label=f"{D['classes'][k]} (AUC={auc(fpr, tpr):.3f})")
        pr, rc, _ = precision_recall_curve(Y[:, k], P[:, k]); axes[1].plot(rc, pr, color=colors[k], lw=2, label=f"{D['classes'][k]} (AP={auc(rc, pr):.3f})")
    axes[0].plot([0, 1], [0, 1], "k--", lw=1); axes[0].set_xlabel("False positive rate"); axes[0].set_ylabel("True positive rate"); axes[0].set_title("ROC (one-vs-rest)")
    axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision"); axes[1].set_title("Precision–Recall")
    for a in axes: a.legend(loc="lower right" if a is axes[0] else "lower left")
    fig.suptitle(f"{PROP} — {D['name']}", fontweight="bold"); savefig(fig, f"Fig07_roc_pr_{key}")

# Fig 8 — reliability diagrams
def reliability(ax, y, P, label, color, n_bins=10):
    conf, acc = P.max(1), (P.argmax(1) == y)
    bins = np.linspace(0, 1, n_bins + 1); xs, ys = [], []
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.sum() > 0: xs.append(conf[m].mean()); ys.append(acc[m].mean())
    ax.plot(xs, ys, "o-", color=color, label=f"{label} (ECE={ece_score(y, P):.3f})")
for key in MEANS:
    R = RUNS[(key, "B")]
    fig, ax = plt.subplots(figsize=(5.8, 5.2)); ax.plot([0, 1], [0, 1], "k--", lw=1, label="Perfect calibration")
    reliability(ax, R["y"], R["oof"]["OCH-MRNet (avg fusion)"], "Branch average (no meta)", "#F39C12")
    reliability(ax, R["y"], R["oof"][PROP], PROP, PROP_COLOR)
    reliability(ax, R["y"], R["oof"][best_baseline(key)], best_baseline(key), BASE_COLOR)
    ax.set_xlabel("Confidence"); ax.set_ylabel("Accuracy"); ax.legend(fontsize=8); ax.set_title(f"Reliability — {DATA[key]['name']}")
    savefig(fig, f"Fig08_reliability_{key}")

# Fig 9 — conformal
for key in MEANS:
    cf = RUNS[(key, "B")]["conf"].groupby(["Method", "Alpha"]).mean(numeric_only=True).reset_index()
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.3))
    for meth, mk in zip(cf.Method.unique(), ["o", "s"]):
        s = cf[cf.Method == meth]
        axes[0].plot(1 - s.Alpha, s.Coverage, marker=mk, label=meth); axes[1].plot(1 - s.Alpha, s.HighClassCoverage, marker=mk, label=meth)
        axes[2].plot(1 - s.Alpha, s.AvgSetSize, marker=mk, label=f"{meth} — set size"); axes[2].plot(1 - s.Alpha, s.Abstain, marker=mk, ls=":", label=f"{meth} — abstain rate")
    for a in axes[:2]: a.plot([0.78, 0.97], [0.78, 0.97], "k--", lw=1, label="Target")
    axes[0].set_title("Marginal coverage"); axes[1].set_title(f"Coverage on '{DATA[key]['classes'][-1]}' patients"); axes[2].set_title("Set size & clinician-referral rate")
    for a in axes: a.set_xlabel("Target coverage (1 − α)"); a.legend(fontsize=7)
    savefig(fig, f"Fig09_conformal_{key}")

# Fig 10 — CD-style rank plot
for key, S in STATS.items():
    rk = S["ranks"]; fig, ax = plt.subplots(figsize=(8, 0.42 * len(rk) + 1.5))
    yy = np.arange(len(rk))[::-1]
    ax.axvspan(rk.iloc[0], rk.iloc[0] + S["CD"], color="#D5F5E3", label=f"Within CD of best (CD={S['CD']:.2f})")
    ax.scatter(rk.values, yy, c=[PROP_COLOR if m == PROP else BASE_COLOR for m in rk.index], s=60, zorder=3)
    ax.set_yticks(yy); ax.set_yticklabels(rk.index); ax.set_xlabel("Average rank (Macro-F1; lower = better)")
    ax.set_title(f"Friedman p={S['friedman'].pvalue:.2e} | Nemenyi critical difference — {DATA[key]['name']}"); ax.legend(loc="lower right")
    savefig(fig, f"Fig10_critical_difference_{key}")

# Fig 12 — Decision curve analysis (highest-risk class vs rest)
def net_benefit(yb, s, th):
    n = len(yb); tp = ((s >= th) & (yb == 1)).sum(); fp = ((s >= th) & (yb == 0)).sum()
    return tp / n - fp / n * th / (1 - th)
for key in MEANS:
    R, D = RUNS[(key, "B")], DATA[key]; yb = (R["y"] == D["high"]).astype(int); th = np.linspace(0.01, 0.8, 80)
    fig, ax = plt.subplots(figsize=(6.5, 4.8))
    for name, col in [(PROP, PROP_COLOR), (best_baseline(key), BASE_COLOR)]:
        ax.plot(th, [net_benefit(yb, R["oof"][name][:, D["high"]], t) for t in th], color=col, lw=2, label=name)
    prev = yb.mean(); ax.plot(th, [prev - (1 - prev) * t / (1 - t) for t in th], "k:", label="Treat all"); ax.axhline(0, color="gray", lw=1, label="Treat none")
    ax.set_ylim(-0.05, prev + 0.05); ax.set_xlabel("Threshold probability"); ax.set_ylabel("Net benefit")
    ax.set_title(f"Decision curve — '{D['classes'][-1]}' detection ({D['name']})"); ax.legend(fontsize=8)
    savefig(fig, f"Fig12_decision_curve_{key}")

# Fig 13 — robustness
for key in MEANS:
    rb = RUNS[(key, "B")]["rob"].groupby(["Model", "Noise"])[["MacroF1", "HighRecall"]].mean().reset_index()
    save_table(rb.round(4), f"T11_robustness_{key}")
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    for m in rb.Model.unique():
        s = rb[rb.Model == m]; kw = dict(color=PROP_COLOR, lw=2.8) if m == PROP else dict(lw=1.4)
        axes[0].plot(s.Noise, s.MacroF1, "o-", label=m, **kw); axes[1].plot(s.Noise, s.HighRecall, "o-", label=m, **kw)
    axes[0].set_ylabel("Macro-F1"); axes[1].set_ylabel(f"Recall '{DATA[key]['classes'][-1]}'")
    for a in axes: a.set_xlabel("Gaussian sensor noise (× feature SD)"); a.legend(fontsize=7)
    fig.suptitle(f"Robustness to measurement noise — {DATA[key]['name']}", fontweight="bold"); savefig(fig, f"Fig13_robustness_{key}")

# Fig 14 — ablation
for key, t in ABL_TABLES.items():
    t2 = t.set_index("Model").drop(index="OCH-MRNet (Full)")
    fig, ax = plt.subplots(figsize=(9, 4.5)); yy = np.arange(len(t2))
    ax.barh(yy - 0.2, t2["ΔMacroF1 vs Full (pp)"], 0.4, label="Δ Macro-F1 (pp)", color="#2874A6")
    ax.barh(yy + 0.2, t2["ΔHighRecall vs Full (pp)"], 0.4, label=f"Δ Recall '{DATA[key]['classes'][-1]}' (pp)", color="#C0392B")
    ax.axvline(0, color="k", lw=1); ax.set_yticks(yy); ax.set_yticklabels(t2.index); ax.legend()
    ax.set_title(f"Ablation: change vs full OCH-MRNet — {DATA[key]['name']}"); savefig(fig, f"Fig14_ablation_{key}")

# Fig 15 + Table 9 — subgroup fairness by maternal age
R, D = RUNS[("maternal", "B")], DATA["maternal"]
bands = pd.cut(R["X"]["Age"], [0, 19, 34, 200], labels=["<20 (teen)", "20–34", "≥35 (advanced)"])
fr_rows = []
for name in [PROP, best_baseline("maternal")]:
    P = R["oof"][name]
    for b in bands.cat.categories:
        m = (bands == b).values
        if m.sum() < 5: continue
        yy, pp = R["y"][m], P[m].argmax(1)
        fr_rows.append(dict(Model=name, AgeBand=b, n=int(m.sum()), n_high=int((yy == 2).sum()), Accuracy=accuracy_score(yy, pp),
                            MacroF1=f1_score(yy, pp, average="macro", labels=[0, 1, 2], zero_division=0),
                            HighRecall=recall_score(yy, pp, labels=[2], average="macro", zero_division=0),
                            UnderTriage=float(np.mean(pp < yy)), ECE=ece_score(yy, P[m])))
fair = pd.DataFrame(fr_rows).round(4); save_table(fair, "T09_subgroup_fairness_age")
print(fair.to_string(index=False))
fig, ax = plt.subplots(figsize=(8, 4.2))
sns.barplot(data=fair, x="AgeBand", y="MacroF1", hue="Model", palette=[PROP_COLOR, BASE_COLOR], ax=ax)
ax.set_ylim(0, 1); ax.set_title("Subgroup performance by maternal age (Macro-F1)"); savefig(fig, "Fig15_subgroup_age")

In [ ]:
# %% [C14] FINAL MODEL (all deduplicated data) -> SHAP + COUNTERFACTUALS ===========================
import shap
FINAL = {}
for key, D in DATA.items():
    set_seed(CFG["SEED"]); FINAL[key] = OCHMRNet(key, D["K"], seed=CFG["SEED"]).fit(D["Xd"], D["yd"], D["gd"])
    print(f"[final] {key}: FT epochs={FINAL[key].ft_epochs_}, CatBoost iters={FINAL[key].cat_iters_}")

SHAP_TABLES = {}
for key, D in DATA.items():
    mdl, Xd = FINAL[key], D["Xd"]; cols = list(Xd.columns)
    f = lambda A: mdl.predict_proba(pd.DataFrame(A, columns=cols))
    rng = np.random.default_rng(CFG["SEED"])
    n_exp = 150 if key == "maternal" else 100
    # stratified explanation sample
    idx = np.concatenate([rng.choice(np.where(D["yd"] == k)[0], min(n_exp // D["K"], (D["yd"] == k).sum()), replace=False) for k in range(D["K"])])
    Xe = Xd.iloc[idx].reset_index(drop=True)
    expl = shap.KernelExplainer(f, shap.kmeans(Xd.values, 15))
    sv = expl.shap_values(Xe.values, nsamples=300 if key == "maternal" else 400, silent=True)
    sv = np.stack(sv, -1) if isinstance(sv, list) else np.asarray(sv)          # -> (n, features, classes)
    ev = np.atleast_1d(expl.expected_value); hi = D["high"]
    # Fig 11a — beeswarm (highest-risk class)
    plt.figure(); shap.summary_plot(sv[:, :, hi], Xe, show=False, max_display=15)
    plt.title(f"SHAP — '{D['classes'][hi]}' class ({D['name']})"); savefig(plt.gcf(), f"Fig11a_shap_beeswarm_{key}")
    # Fig 11b — mean |SHAP| per class
    imp = pd.DataFrame(np.abs(sv).mean(0), index=cols, columns=D["classes"]); imp["Total"] = imp.sum(1)
    imp = imp.sort_values("Total", ascending=True).tail(15)
    fig, ax = plt.subplots(figsize=(7.5, 0.4 * len(imp) + 1.5))
    imp[D["classes"]].plot.barh(stacked=True, ax=ax, color=["#27AE60", "#F39C12", "#C0392B"])
    ax.set_xlabel("mean |SHAP value|"); ax.set_title(f"Global feature importance — {D['name']}"); savefig(fig, f"Fig11b_shap_importance_{key}")
    SHAP_TABLES[key] = imp.sort_values("Total", ascending=False).round(5).reset_index().rename(columns={"index": "Feature"})
    save_table(SHAP_TABLES[key], f"T12_shap_importance_{key}")
    # Fig 11c — dependence for top-2 features
    top2 = SHAP_TABLES[key].Feature.iloc[:2].tolist()
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
    for ax, ft in zip(axes, top2):
        j = cols.index(ft); sc = ax.scatter(Xe[ft], sv[:, j, hi], c=sv[:, :, hi].sum(1), cmap="coolwarm", s=22)
        ax.axhline(0, color="gray", lw=0.8); ax.set_xlabel(ft); ax.set_ylabel(f"SHAP for '{D['classes'][hi]}'")
    plt.colorbar(sc, ax=axes, label="total SHAP (high class)"); fig.suptitle(f"SHAP dependence — {D['name']}", fontweight="bold")
    fig.savefig(os.path.join(FIG, f"Fig11c_shap_dependence_{key}.png"), dpi=300, bbox_inches="tight"); plt.show(); plt.close(fig)
    # Fig 11d — waterfall for one correctly-identified high-risk patient
    Pe = f(Xe.values); cand = np.where((D["yd"][idx] == hi) & (Pe.argmax(1) == hi))[0]
    if len(cand):
        i = cand[0]
        e = shap.Explanation(values=sv[i, :, hi], base_values=ev[hi] if len(ev) > 1 else ev[0], data=Xe.iloc[i].values, feature_names=cols)
        plt.figure(); shap.plots.waterfall(e, show=False, max_display=12)
        plt.title(f"Local explanation — patient predicted '{D['classes'][hi]}' (p={Pe[i, hi]:.2f})"); savefig(plt.gcf(), f"Fig11d_shap_waterfall_{key}")

# ---- Counterfactual explanations (maternal; Age is immutable) — custom search, no extra package needed
D, mdl = DATA["maternal"], FINAL["maternal"]; Xd = D["Xd"]
MUTABLE = {"SystolicBP": (90, 160), "DiastolicBP": (60, 100), "BS": (6.0, 19.0), "BodyTemp": (98.0, 103.0), "HeartRate": (60, 90)}
sd = Xd.std(); rng = np.random.default_rng(CFG["SEED"])
Pall = mdl.predict_proba(Xd); hi_idx = np.where((D["yd"] == 2) & (Pall.argmax(1) == 2))[0]
cf_rows = []
for pid in rng.choice(hi_idx, min(6, len(hi_idx)), replace=False):
    x0 = Xd.iloc[pid]; cands = []
    for _ in range(4000):
        x = x0.copy(); feats = rng.choice(list(MUTABLE), rng.integers(1, 4), replace=False)
        for ft in feats:
            lo, hi_ = MUTABLE[ft]; x[ft] = rng.uniform(lo, hi_)
            x[ft] = round(x[ft], 1) if ft in ("BS", "BodyTemp") else round(x[ft])
        if x["DiastolicBP"] >= x["SystolicBP"]: continue
        cands.append(x)
    C = pd.DataFrame(cands); Pc = mdl.predict_proba(C)
    ok = Pc.argmax(1) < 2
    if not ok.any(): continue
    C, Pc = C[ok].reset_index(drop=True), Pc[ok]
    keep = ~C.duplicated(subset=list(MUTABLE)).values                 # unique counterfactuals only
    C, Pc = C[keep].reset_index(drop=True), Pc[keep]
    dist = (np.abs(C[list(MUTABLE)] - x0[list(MUTABLE)]) / sd[list(MUTABLE)]).sum(1).values
    for rank, j in enumerate(np.argsort(dist)[:3]):
        ch = {ft: f"{x0[ft]:g} → {C.loc[j, ft]:g}" for ft in MUTABLE if C.loc[j, ft] != x0[ft]}
        cf_rows.append(dict(PatientID=int(pid), Age=x0["Age"], Original_P_high=round(Pall[pid, 2], 3), CF_rank=rank + 1,
                            Changes="; ".join(f"{k}: {v}" for k, v in ch.items()), New_class=D["classes"][Pc[j].argmax()],
                            New_P_high=round(Pc[j, 2], 3), Norm_L1_distance=round(dist[j], 3)))
cf_df = pd.DataFrame(cf_rows); save_table(cf_df, "T13_counterfactuals_maternal"); print(cf_df.to_string(index=False))

In [ ]:
# %% [C15] LITERATURE COMPARISON + FINAL SUMMARY (paste this output back) ===========================
LIT = pd.DataFrame([
    ("Monson & Sabarmathi (stacking RF+XGB+GB)", "Maternal (1014 raw)", "Random split, duplicates kept", 0.9113, np.nan, np.nan),
    ("Yesmin et al., 2026 (fuzzy-XGBoost)", "Maternal (1014 raw)", "As reported", 0.8867, np.nan, 0.9703),
    ("Solent study (Random Forest + SMOTE)", "Maternal (1014 raw)", "80/20 split, SMOTE after split", 0.882, 0.888, 0.966),
], columns=["Study", "Data", "Protocol", "Accuracy", "MacroF1", "AUC"])
mB = MEANS["maternal"].loc[PROP]; mA = A[A.Model == PROP][["Accuracy", "MacroF1", "AUC"]].mean()
LIT.loc[len(LIT)] = [f"{PROP} — Protocol A (same naive protocol)", "Maternal (1014 raw)", "Naive stratified CV", mA.Accuracy, mA.MacroF1, mA.AUC]
LIT.loc[len(LIT)] = [f"{PROP} — Protocol B (leakage-free)", "Maternal (452 unique)", "Dedup + group-stratified 5×5 CV", mB.Accuracy, mB.MacroF1, mB.AUC]
save_table(LIT.round(4), "T14_literature_comparison")
print("NOTE: verify the literature numbers against the original papers before submission.\n")

print("=" * 100); print("FINAL SUMMARY — copy everything below and share"); print("=" * 100)
print(audit_df.to_string(index=False))
print("\n--- LEAKAGE (maternal) ---"); print(leak[["Model", "Acc_A (naive)", "Acc_B (dedup+group)", "ΔAcc (inflation, pp)", "ΔF1 (inflation, pp)"]].to_string(index=False))
for key in MEANS:
    print(f"\n--- MAIN ({key}) ---"); print(MAIN_TABLES[key][["Model", "Accuracy", "MacroF1", "MCC", "QWK", "AUC", "HighRecall", "UnderTriage", "ECE", "Brier"]].to_string(index=False))
    print(f"\n--- ABLATION ({key}) ---"); print(ABL_TABLES[key][["Model", "MacroF1_mean", "HighRecall_mean", "ECE_mean", "ΔMacroF1 vs Full (pp)", "ΔHighRecall vs Full (pp)"]].to_string(index=False))
    cf = RUNS[(key, "B")]["conf"].groupby(["Method", "Alpha"])[["Coverage", "HighClassCoverage", "AvgSetSize", "Abstain", "SingletonAcc"]].mean().round(3)
    print(f"\n--- CONFORMAL ({key}) ---"); print(cf.to_string())
    S = STATS[key]; print(f"\n--- STATS ({key}) --- Friedman p={S['friedman'].pvalue:.2e}, CD={S['CD']:.3f}"); print(S["ranks"].round(2).to_string())
    print(f"\n--- SHAP top features ({key}) ---"); print(SHAP_TABLES[key].head(8).to_string(index=False))
print("\n--- FAIRNESS (age) ---"); print(fair.to_string(index=False))
print("\n--- COUNTERFACTUALS ---"); print(cf_df.head(9).to_string(index=False))
print("\n--- LITERATURE ---"); print(LIT.round(4).to_string(index=False))
print("\nFigures:", sorted(os.listdir(FIG))); print("Tables:", sorted(os.listdir(TAB)))

# zip everything for download
import shutil; shutil.make_archive("/kaggle/working/och_mrnet_outputs", "zip", CFG["OUT"])
print("\nDownload: /kaggle/working/och_mrnet_outputs.zip")